Testing how to get node count from skeleton data with Skan package

In [ ]:
from glob import glob
import skimage as sk
from skan import draw
from skan.csr import make_degree_image
import napari
import pandas as pd
from tqdm import tqdm

In [ ]:
def load_images(path):
    '''
    read in images and get names

    Args:
        path: parent path of data

    Returns:
        names: list of file names
        images: list of images read in as nd arrays
    '''
    files = sorted(glob(os.path.join(path,'Skeletons','*.tif')))
    images = list(map(sk.io.imread,files))
    names = list(map(os.path.basename,files))
    if 'Nov' in path:
        names = [name.replace('Skeleton_Cropped_masked_', '') for name in names]
    elif 'Oct' in path:
        names = [name.replace('_skeleton','') for name in names]
    else:
        print('Path not matched to either condition')
    return names, images

def get_nodes(images):
    '''
    create degree images and threshold to get nodes

    Args:
        images: list of skeleton image arrays

    Returns:
        labeled_nodes: list of nd arrays of labeled nodes
    '''
    degree_images = list(map(make_degree_image,images))
    nodes = [degree_image > 2 for degree_image in degree_images]
    labeled_nodes = list(map(sk.measure.label,nodes))
    return labeled_nodes

def count_nodes(labeled_nodes,images,names):
    '''
    count number of nodes per skeleton, create data frame

    Args:
        labeled_nodes: list of labeled nodes as arrays
        images: list of original skeleton images as arrays

    Returns:
        merged_df: pandas data frame with each node as a row and parent skeleton as a column
        labeled_skeletons: list of labeled skeletons as arrays   
    '''
    props = ['label','intensity_max']
    dfs = []
    labeled_skeletons = list(map(sk.measure.label,images))
    for nodes,skeletons,name in zip(labeled_nodes,labeled_skeletons,names):
        df = sk.measure.regionprops_table(label_image=nodes,intensity_image=skeletons,properties=props)
        df = pd.DataFrame.from_dict(df)
        df['image_name'] = name[:-4]
        dfs.append(df)
    merged_df = pd.concat(dfs,ignore_index=True)
    renamed_df = merged_df.rename(columns={'label':'node_ID','intensity_max':'skeleton_ID'})
    return renamed_df, labeled_skeletons

def save_things(labeled_nodes,labeled_skeletons,merged_df,names,path):
    '''
    Save the labeled node images and the data frames to the right places

    Args:
        labeled_nodes: list of labeled nodes arrays
        labeled_skeletons: list of labeled skeleton arrays
        merged_df: data frame of nodes and parent skeletons
        names: list of image names
        path: parent folder of data
    '''
    os.makedirs(os.path.join(path,'Nodes'),exist_ok=True)
    os.makedirs(os.path.join(path,'Labeled_Skeletons'),exist_ok=True)

    if '50' in path:
        condition = '50mM'
    elif '90_min' in path:
        condition = '300mM_90min'
    else:
        condition = '300mM_6hr'
    
    merged_df.to_csv(os.path.join(path,'Measurements','Node_counts_'+condition+'.csv'))
    for nodes, skeletons, name in zip(labeled_nodes,labeled_skeletons,names):
        sk.io.imsave(os.path.join(path,'Nodes','Skeleton_Nodes_'+name[:-4]+'.tif'),nodes,check_contrast=False)
        sk.io.imsave(os.path.join(path,'Labeled_Skeletons','Labeled_Skeletons_'+name[:-4]+'.tif'),skeletons,check_contrast=False)


In [ ]:
nov_na_50 = r'Image_Data\Nov_2025_300mm_exp\50_mm'
nov_na_300_90min = r'Image_Data\Nov_2025_300mm_exp\300_mm_90_min'
oct_na_50 = r'Image_Data\Oct_2025_300mm_exp\50_mM'
oct_na_300_90min = r'Image_Data\Oct_2025_300mm_exp\300_mM_1_hr'
oct_na_300_6hr = r'Image_Data\Oct_2025_300mm_exp\300_mM_6_hr'
paths = [nov_na_300_90min,nov_na_50,oct_na_300_90min,oct_na_300_6hr,oct_na_50]

In [ ]:
for path in tqdm(paths):
    names, images = load_images(path)
    labeled_nodes = get_nodes(images)
    merged_df, labeled_skeletons = count_nodes(labeled_nodes,images,names)
    save_things(labeled_nodes,labeled_skeletons,merged_df,names,path)

## Testing that connected components in scikit-imge and scipy assign labels in the same way
Want to make sure that the node data created with scikit-image can be combined with the skeleton dataframe based on the skeleton ID

In [ ]:
#comparing scipy connected components
skellies_csgraph = csr_array(test_skellie)
#components, scipy_labels = connected_components(csgraph=skellies_csgraph,directed=False,return_labels=True)

In [ ]:
bool_skellies = test_skellie/255

In [ ]:
print(np.max(bool_skellies))

In [ ]:
#comparing scipy connected components
test_skellie_data = summarize(Skeleton(test_skellie, spacing=1),separator="_")


In [ ]:
one_skellie = test_skellie_data[test_skellie_data['skeleton_id'] == 15]

In [ ]:
one_skellie.head()

In [ ]:
draw.overlay_euclidean_skeleton_2d(test_skellie,one_skellie)